# Clean, Source-Organized Copy
Saved outputs and machine-specific metadata are intentionally excluded. The byte-identical original remains in the local audit archive. Prepare the local workbench before running this notebook. Notebooks 01 and 02 execute GPU inference when explicitly enabled; 03 disables its optional GPU activation check by default.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts" / "lab.py").is_file()
)
WORKSPACE = PROJECT_ROOT / ".local" / "workbench"
if not (WORKSPACE / "weight_edit.py").is_file():
    raise RuntimeError("Run python scripts/lab.py prepare --with-local-data first.")
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))

ALLOW_GPU_INFERENCE = False
if not ALLOW_GPU_INFERENCE:
    raise RuntimeError('Set ALLOW_GPU_INFERENCE=True only to deliberately run the GPU demonstration.')


# Weight-space programming: local starting point

Load the original Qwen3.5-0.8B, inspect its editable weights, and generate a short response on the GPU. This notebook does not apply any edits yet.

Reference: https://rikiyat.github.io/blog/weight-space-programming/

Select the **Python (weight-space)** kernel. All model loading below is local-only; no model code is fetched or trusted.

In [ ]:
from pathlib import Path

import torch
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration

print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select the project kernel and check the NVIDIA driver.')
print('GPU:', torch.cuda.get_device_name(0))

## Load the pinned, original checkpoint

Run Jupyter from the project root. The full release includes a vision tower, so use its correct conditional-generation class even though this experiment only supplies text. Keep BF16 weights rather than quantizing them.

In [ ]:
model_dir = Path.cwd() / 'models' / 'Qwen3.5-0.8B'
if not (model_dir / 'config.json').is_file():
    raise FileNotFoundError('Run download_model.py from this project first, and launch Jupyter in the project root.')

tokenizer = AutoTokenizer.from_pretrained(
    model_dir, local_files_only=True, trust_remote_code=False
)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    model_dir,
    dtype=torch.bfloat16,
    local_files_only=True,
    trust_remote_code=False,
    use_safetensors=True,
).to('cuda').eval()

print('Device:', model.device)
print('Unique parameters:', f'{sum(p.numel() for p in model.parameters()):,}')
print('Allocated GPU memory (GiB):', round(torch.cuda.memory_allocated() / 1024**3, 2))

In [ ]:
# These are the two MLP matrices targeted by the article's published edit.
for layer_index in (15, 19):
    name = f'model.language_model.layers.{layer_index}.mlp.down_proj.weight'
    weight = model.get_parameter(name)
    print(name)
    print('  shape:', tuple(weight.shape), 'dtype:', weight.dtype)
    print('  norm:', round(weight.detach().float().norm().item(), 4))

## Short inference smoke test

Thinking is disabled and output is capped for a quick setup check. This is not a math benchmark and does not reproduce the article's long-context evaluation settings.

In [ ]:
messages = [{'role': 'user', 'content': 'Say hello in one short sentence.'}]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=48, do_sample=False)

new_tokens = generated[0, inputs['input_ids'].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)
if not response.strip():
    raise RuntimeError('The inference smoke test returned an empty response.')
print(response)
print('Generated tokens:', new_tokens.numel())
print('Peak GPU memory (GiB):', round(torch.cuda.max_memory_allocated() / 1024**3, 2))

## Next experiment

Build a fixed baseline problem set, replay the authors' hash-verified sparse edit, and compare repetition, token-limit hits, and correctness. Use separate held-out problems before drawing conclusions. Restart the kernel to release GPU memory before opening another model notebook.